In [3]:
# Please execute/shift-return this cell everytime you run the notebook.  Don't edit it. 
%load_ext autoreload
%autoreload 2
from notebook import * 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Matrix tiling algorithm with Transpositon/Register/Rectangular


In [7]:
render_code("matrix_mul/rect_blockmm_trans.c", show=["//START","//END"])

// matrix_mul/rect_blockmm_trans.c:73-96 (24 lines)
//START
void blockmm_transpose(double **a, double **b, double **c, uint64_t M, uint64_t N, uint64_t K)
{
  int i,j,k, ii, jj, kk;
  double result=0;
  for(i = 0; i < M; i+=tile_size_y)
  {
    for(j = 0; j < K; j+=tile_size_y)
    {
      for(k = 0; k < N; k+=tile_size_x)
      {
          for(ii = i; ii < i+tile_size_y; ii++)
              for(jj = j; jj < j+tile_size_y; jj++)
              {
                      result = 0;
                      for(kk = k; kk < k+tile_size_x ; kk++)
                          result += a[ii][kk]*b[jj][kk];
                      c[ii][jj] += result;
              }
      }
    }
  }  
}
//END

In [25]:
display_df_mono(render_csv("matrix_mul/blockmm_transpose_reg.csv"))

,index,size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,2048,4,88420275497,19260798162,0.217832,0.179103,3.449672,0.012464,323402802,25947195171
1,1,2048,8,60794386700,16316765393,0.268393,0.179153,2.923191,0.006240,99179001,15893700615
2,2,2048,16,49275504187,10252678197,0.208068,0.179006,1.835293,0.062430,750102701,12015109307
3,3,2048,32,43909426178,9882611247,0.225068,0.179029,1.769276,0.101084,1038323461,10271847097
4,4,2048,64,41302516469,10209904883,0.247198,0.179073,1.828317,0.082142,775254954,9437961457
5,5,2048,128,40017565663,11807556573,0.295059,0.179148,2.115303,0.061286,553401332,9029855550
6,6,2048,256,39381375103,14038809390,0.356483,0.179092,2.514241,0.020085,177324389,8828627936
7,7,2048,512,39066967651,16255933938,0.416104,0.179067,2.910896,0.013996,122176548,8729410035


In [8]:
! cd matrix_mul; make rect_blockmm_trans; echo "size,tile_size_x,tile_size_y,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > rect_blockmm_trans.csv
for tile_x in (4, 8, 16, 32, 64, 128, 256, 512):
    for tile_y in (4, 8, 16, 32, 64, 128, 256, 512):
        ! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 {tile_x} {tile_y} >> ./matrix_mul/rect_blockmm_trans.csv 

make: 'rect_blockmm_trans' is up to date.
10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102

In [9]:
display_df_mono(render_csv("matrix_mul/rect_blockmm_trans.csv"))

,index,size,tile_size_x,tile_size_y,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,2048,4,4,88093305081,14569520878,0.165387,0.218068,3.177146,0.012145,311566946,25653312306
1,1,2048,4,8,82710986573,13565478324,0.164011,0.179102,2.429606,0.007559,174348124,23064216241
2,2,2048,4,16,80833708081,14709788909,0.181976,0.179101,2.634538,0.012448,275726929,22150387708
3,3,2048,4,32,80099498172,16653432156,0.207909,0.179107,2.982753,0.009899,215691166,21789005705
4,4,2048,4,64,79777936509,15724435572,0.197103,0.179035,2.815226,0.006626,143329898,21630130260
5,5,2048,4,128,79630676723,15175121734,0.190569,0.179296,2.720842,0.004883,105254614,21556960409
6,6,2048,4,256,79558116008,14464155606,0.181806,0.179094,2.590443,0.004567,98278126,21521122146
7,7,2048,4,512,79523256120,13881407209,0.174558,0.179049,2.485449,0.034114,733585453,21503738923
8,8,2048,8,4,63384498767,11628947863,0.183467,0.179153,2.083364,0.003597,61609277,17125811882
9,9,2048,8,8,60695445035,11673416088,0.192328,0.179419,2.094428,0.003877,61386334,15831818242


## Prefetch

x86 provide prefetch instructions. As a programmer, you may insert ```_mm_prefetch``` in x86 programs to perform software prefetch for your code. The gcc compiler also has a flag ```-fprefetch-loop-arrays``` to automatically insert software prefetch instructions.

### Using prefetch in matrix transpose code

The following example is a highly optimized matrix transpose code. In the example, we try to prefetch the next row.

In [ ]:
render_code("./prefetch/transpose.cpp", lang="c++", show=["//START", "//END"])

Now, let's take a look of what's happening!

In [ ]:
! cd prefetch; make clean; make
# ! echo "Without prefetch -- the baseline"; ssh htseng@celebi "lscpu | grep Model; cd courses/CS203/demo/memory/prefetch/; ./transpose"
! echo "Without prefetch -- the baseline"
! lscpu | grep Model
! ./prefetch/transpose
! echo "With prefetch"
! ./prefetch/transpose_prefetch

Let's try a different machine now.

In [ ]:
! ssh htseng@xerneas "cd /nfshome/htseng/courses/CSE142/demo/software_optimizations_memory/; make -C ./prefetch clean; make -C ./prefetch ; lscpu | grep Model"
! echo "Without prefetch -- the baseline"; ssh htseng@xerneas  "/nfshome/htseng/courses/CSE142/demo/software_optimizations_memory/prefetch/transpose"
! echo "With prefetch";  ssh htseng@xerneas  "/nfshome/htseng/courses/CSE142/demo/software_optimizations_memory/prefetch/transpose_prefetch"

In [ ]:
! ssh htseng@blissey "cd /nfshome/htseng/courses/CSE142/demo/memory/; make -C ./prefetch clean; make -C ./prefetch ; lscpu | grep Model"
! echo "Without prefetch -- the baseline"; ssh htseng@blissey  "/nfshome/htseng/courses/CSE142/demo/memory/prefetch/transpose"
! echo "With prefetch";  ssh htseng@blissey  "/nfshome/htseng/courses/CSE142/demo/memory/prefetch/transpose_prefetch"

In [ ]:
! ssh htseng@eevee "cd /nfshome/htseng/courses/CSE142/demo/memory/; make -C ./prefetch clean; make -C ./prefetch ; lscpu | grep Model"
! echo "Without prefetch -- the baseline"; ssh htseng@eevee  "/nfshome/htseng/courses/CSE142/demo/memory/prefetch/transpose"
! echo "With prefetch";  ssh htseng@eevee  "/nfshome/htseng/courses/CSE142/demo/memory/prefetch/transpose_prefetch"


-- It doesn't work always!

In [ ]:
render_code("matrix_mul/blockmm_interchange.c", show=["//START","//END"])

In [23]:
! cd matrix_mul; rm -f blockmm_interchange; make blockmm_interchange; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm_interchange.csv
for tile_x in (4, 8, 16, 32, 64, 128, 256, 512):
    ! ./matrix_mul/blockmm_interchange 2048 {tile_x} >> ./matrix_mul/blockmm_interchange.csv 
display_df_mono(render_csv("matrix_mul/blockmm_interchange.csv"))

gcc -O4 -DHAVE_LINUX_PERF_EVENT_H blockmm_interchange.c perfstats.c -o blockmm_interchange
10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,

,index,size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,2048,4,110531516080,27745632026,0.251020,0.199309,5.529964,0.024715,962625605,38949198157
1,1,2048,8,68607718075,18074813184,0.263452,0.179198,3.238972,0.013721,330295850,24073072403
2,2,2048,16,50678263749,13976764593,0.275794,0.179265,2.505548,0.078124,1412824668,18084312900
3,3,2048,32,42341163582,10804534850,0.255178,0.179199,1.936157,0.078895,1214647824,15395836701
4,4,2048,64,38313404486,8898546377,0.232257,0.179263,1.595181,0.079146,1117631832,14121060773
5,5,2048,128,36332966961,7910294071,0.217717,0.179020,1.416101,0.079961,1079489338,13500222451
6,6,2048,256,35351600426,7637093224,0.216032,0.179280,1.369177,0.080086,1056671642,13194202901
7,7,2048,512,34863917519,7902868451,0.226678,0.179121,1.415570,0.063705,830875550,13042471134


In [22]:
! cd matrix_mul; rm -f blockmm_transpose_prefetch; make blockmm_transpose_prefetch; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm_transpose_prefetch.csv
for tile_x in (4, 8, 16, 32, 64, 128, 256, 512):
    ! ./matrix_mul/blockmm_transpose_prefetch 2048 {tile_x} >> ./matrix_mul/blockmm_transpose_prefetch.csv 
display_df_mono(render_csv("matrix_mul/blockmm_transpose_prefetch.csv"))

gcc -O4 -DHAVE_LINUX_PERF_EVENT_H -fprefetch-loop-arrays blockmm_transpose_prefetch.c perfstats.c -o blockmm_transpose_prefetch


,index,size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,2048,4,119805025262,21964733679,0.183337,0.179270,3.937608,0.000512,20302492,39627479507
1,1,2048,8,86490888839,17282452031,0.199818,0.179181,3.096679,0.000964,30099329,31234414800
2,2,2048,16,72651103830,14183173540,0.195223,0.179142,2.540806,0.003074,86745665,28218189546
3,3,2048,32,66132118862,13456496838,0.203479,0.179935,2.421300,0.001295,34825390,26883370136
4,4,2048,64,63227833897,12693913280,0.200765,0.179073,2.273137,0.000839,22123640,26363221523
5,5,2048,128,61727214937,13885201530,0.224945,0.179436,2.491506,0.000367,9564237,26084292576
6,6,2048,256,60966029138,15241313400,0.249997,0.179388,2.734110,0.000297,7711695,25939944573
7,7,2048,512,60589899799,16296783317,0.268969,0.179372,2.923191,0.000131,3401278,25869619052
